<a href="https://colab.research.google.com/github/saifinuha/data-science-2026/blob/main/Pertemuan10_Saifin_Nuha_240401010257.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
import kagglehub

# Download latest version
path = kagglehub.dataset_download("blastchar/telco-customer-churn")

print(path)
df = pd.read_csv("/kaggle/input/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(df.shape)
print(df["Churn"].value_counts(normalize=True))

Using Colab cache for faster access to the 'telco-customer-churn' dataset.
/kaggle/input/telco-customer-churn
(7043, 21)
Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64


In [13]:
from sklearn.model_selection import train_test_split

# Pisahkan X (fitur) dan y (target = Churn)
y = df['Churn']
X = df.drop(columns=['customerID', 'Churn'])

# encoding fitur kategorikal (mis. pd.get_dummies)
X = pd.get_dummies(X, dtype=int)

X_tr, X_te, y_tr, y_te = train_test_split(
              X, y, test_size=0.2, stratify=y, random_state=42)

In [14]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300, class_weight="balanced", random_state=42)
rf.fit(X_tr, y_tr)

RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

In [15]:
from sklearn.metrics import classification_report, roc_auc_score

# Hitung prediksi dan probabilitas
y_pred = rf.predict(X_te)
y_pred_proba = rf.predict_proba(X_te)[:, 1] # Probability of the positive class (Churn = 'Yes')

# Tampilkan classification_report
print("Classification Report:")
print(classification_report(y_te, y_pred))

# Tampilkan ROC-AUC
print(f"ROC-AUC Score: {roc_auc_score(y_te, y_pred_proba)}")

Classification Report:
              precision    recall  f1-score   support

          No       0.83      0.88      0.85      1035
         Yes       0.60      0.49      0.54       374

    accuracy                           0.78      1409
   macro avg       0.71      0.69      0.70      1409
weighted avg       0.77      0.78      0.77      1409

ROC-AUC Score: 0.8204590663669947


In [16]:
# Hitung probabilitas churn (predict_proba) for the test set
churn_probabilities = pd.DataFrame({'Actual_Churn': y_te, 'Predicted_Churn_Prob': y_pred_proba})
display(churn_probabilities.head())

,Actual_Churn,Predicted_Churn_Prob
437,No,0.000000
2280,No,0.626667
2235,No,0.050000
4460,No,0.323333
3761,No,0.020000


## Kesimpulan

Model Random Forest Classifier mencapai akurasi 78% dengan ROC-AUC 0.82. Model ini menunjukkan kemampuan yang cukup baik dalam mengidentifikasi pelanggan yang akan churn, meskipun performanya lebih tinggi untuk kelas non-churn. Dengan adanya probabilitas churn yang dihitung, perusahaan dapat mengidentifikasi pelanggan berisiko tinggi untuk intervensi proaktif. Untuk meningkatkan performa, bisa dipertimbangkan teknik _feature engineering_ lebih lanjut, tuning _hyperparameter_, atau eksplorasi model lain.